# 🔍 Assistente RAG com interface web (Gradio) — Haja Coração (Grupo 4)
**Atividade prática — Engenharia de Recuperação Semântica**

Este notebook monta um **ambiente completo para o usuário**: uma aplicação web onde ele envia arquivos, os documentos ficam gravados no banco e as respostas aparecem num espaço próprio, com gráficos e tabelas quando a pergunta é estatística.

```
           ┌──────────────── INTERFACE (Gradio) ────────────────┐
           │  📁 Documentos   💬 Perguntar   🗄️ Base             │
           └───────┬──────────────────┬─────────────────────────┘
                   │ upload           │ pergunta
                   ▼                  ▼
   PDF/DOCX/DOC/TXT ─► fragmentos ─► ChromaDB  ◄─── Etapa 1: bi-encoder (HNSW)
   CSV/TSV/XLSX ─────► DataFrame ─► disco         Etapa 2: cross-encoder
                   └─► linhas ─────► ChromaDB      LLM local responde
                                                    │
                       roteador: pergunta estatística? ─► pandas + gráfico
```

## Ordem de execução
1. **Ambiente de execução → Alterar o tipo → GPU (T4).**
2. Rode as células **1 a 7** em ordem. A célula 7 mostra a aplicação e gera um **link público** (`https://….gradio.live`) que os alunos podem abrir no navegador ou no celular.
3. Na aplicação: aba **Documentos** → envie arquivos → aba **Perguntar**.

| Aba | O que faz |
|---|---|
| 📁 **Documentos** | Envia PDF, DOCX, DOC, TXT, CSV, TSV ou XLSX e grava no banco; mostra as colunas detectadas e perguntas sugeridas |
| 💬 **Perguntar** | Espaço da resposta: texto, gráfico, tabela de resultado e código pandas equivalente |
| 🗄️ **Base** | Lista o que está gravado e permite remover arquivos |

## Célula 1 — Instalação

In [12]:
# ==============================================================================
# CÉLULA 1 — INSTALAÇÃO (rodar uma vez por sessão do Colab)
# ==============================================================================
!pip install -q chromadb sentence-transformers gradio pypdf python-docx openpyxl requests
!apt-get -qq install -y antiword > /dev/null   # leitor de .doc (Word 97-2003)

## Célula 2 — Configuração
Escolha aqui os modelos (todos **open source**, sem custo por token) e se o banco deve ficar no **Google Drive**.

> 💡 **Persistência:** com `USAR_GOOGLE_DRIVE = False`, os dados ficam no disco do Colab e **somem** quando a sessão termina. Com `True`, ficam no seu Drive e reaparecem na próxima aula.
>
> ⚠️ Ao trocar o **bi-encoder**, os vetores antigos não servem mais (têm outro tamanho). Por isso o nome da coleção inclui o nome do modelo: cada encoder tem sua própria coleção.

In [13]:
# ==============================================================================
# CÉLULA 2 — CONFIGURAÇÃO: modelos, persistência e parâmetros
# ------------------------------------------------------------------------------
# Tudo que a turma pode trocar fica aqui. Todos os modelos são open source e rodam
# LOCALMENTE (sem chave de API e sem custo por token).
# ==============================================================================
import os, torch

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Dispositivo: {dispositivo.upper()}")

# ---------- PERSISTÊNCIA ----------
# O disco do Colab é apagado quando a sessão termina. Para guardar o banco de verdade,
# use o Google Drive (vai pedir autorização na primeira vez).
USAR_GOOGLE_DRIVE = False
if USAR_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA_BASE = "/content/drive/MyDrive/rag_aula"
else:
    PASTA_BASE = "./rag_aula"
CAMINHO_BANCO = os.path.join(PASTA_BASE, "chroma")     # vetores (ChromaDB + SQLite)
PASTA_TABELAS = os.path.join(PASTA_BASE, "tabelas")    # tabelas completas (pandas)
os.makedirs(CAMINHO_BANCO, exist_ok=True)
os.makedirs(PASTA_TABELAS, exist_ok=True)

# ---------- ETAPA 1: BI-ENCODER (embeddings) ----------
# Opções open source (descomente uma). Os prefixos são exigidos por alguns modelos.
MODELO_BI_ENCODER, PREFIXO_CONSULTA, PREFIXO_DOCUMENTO = (
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", "", "")   # leve, 384 dim
# MODELO_BI_ENCODER, PREFIXO_CONSULTA, PREFIXO_DOCUMENTO = (
#     "intfloat/multilingual-e5-base", "query: ", "passage: ")               # 768 dim, MIT
# MODELO_BI_ENCODER, PREFIXO_CONSULTA, PREFIXO_DOCUMENTO = (
#     "BAAI/bge-m3", "", "")                                                 # 1024 dim, forte, mais pesado

# ---------- ETAPA 2: CROSS-ENCODER (re-ranking) ----------
MODELO_CROSS_ENCODER = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"   # multilíngue (mMARCO)
# MODELO_CROSS_ENCODER = "BAAI/bge-reranker-v2-m3"                     # mais forte, mais pesado
# MODELO_CROSS_ENCODER = None                                          # desliga o re-ranking

# ---------- GERAÇÃO: LLM ----------
# DeepSeek-R1 via Ollama. Execute a célula opcional do fim do notebook
# ANTES de carregar os modelos para instalar o Ollama e baixar o modelo.
BACKEND_LLM = "ollama"
MODELO_LLM = None
MODELO_OLLAMA = "deepseek-r1:1.5b"

# Limiares conservadores para evitar respostas com contexto pouco relacionado.
# Ajuste após testar perguntas reais do projeto.
MIN_SIMILARIDADE = 0.25
MIN_RELEVANCIA = 0.30

# ---------- PARÂMETROS DO RAG ----------
K_CANDIDATOS = 10        # etapa 1: rede ampla
K_CONTEXTO = 3           # etapa 2: no máximo 3 trechos vão para a LLM
LIMIAR_RELATIVO = 0.60   # trecho só entra se tiver ≥ 60% da nota do 1º colocado

🚀 Dispositivo: CUDA


In [14]:
# ==============================================================================
# OLLAMA - execute apos a configuracao (celula 2), antes de carregar os modelos.
# ==============================================================================
import subprocess
import sys
import time
import requests

if BACKEND_LLM == "ollama":
    subprocess.run(["apt-get", "-qq", "install", "-y", "zstd", "pciutils"],
                   check=True, stdout=subprocess.DEVNULL)
    instalador = subprocess.run(["curl", "-fsSL", "https://ollama.com/install.sh"],
                                check=True, capture_output=True)
    subprocess.run(["sh"], input=instalador.stdout, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ollama"], check=True)

    OLLAMA_API = "http://127.0.0.1:11434"

    def verificar_ollama():
        try:
            resposta = requests.get(f"{OLLAMA_API}/api/tags", timeout=3)
            resposta.raise_for_status()
            return resposta.json()
        except requests.RequestException:
            return None

    status_ollama = verificar_ollama()
    if status_ollama is None:
        processo_ollama = subprocess.Popen(
            ["ollama", "serve"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        prazo = time.monotonic() + 120
        while time.monotonic() < prazo:
            if processo_ollama.poll() is not None:
                raise RuntimeError(
                    "O processo 'ollama serve' encerrou antes de iniciar. "
                    "Verifique se a porta 11434 esta livre e execute esta celula novamente."
                )
            status_ollama = verificar_ollama()
            if status_ollama is not None:
                break
            time.sleep(2)

    if status_ollama is None:
        raise RuntimeError(
            f"O Ollama nao respondeu em {OLLAMA_API} apos 120 segundos. "
            "Execute esta celula novamente e verifique a saida do servidor."
        )

    subprocess.run(["ollama", "pull", MODELO_OLLAMA], check=True)
    print(f"Ollama pronto em {OLLAMA_API}; modelo {MODELO_OLLAMA} baixado.")

Ollama pronto em http://127.0.0.1:11434; modelo deepseek-r1:1.5b baixado.


## Célula 3 — Carregamento dos modelos e do banco
Na primeira execução, os modelos são baixados do Hugging Face (alguns minutos).

In [15]:
# ==============================================================================
# CÉLULA 3 — CARREGAMENTO DOS MODELOS E DO BANCO
# ==============================================================================
import io, re, csv, json, html, time, hashlib, difflib, unicodedata, subprocess, tempfile
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder

print("[1/3] Bi-encoder...")
bi_encoder = SentenceTransformer(MODELO_BI_ENCODER, device=dispositivo)

print("[2/3] Cross-encoder...")
cross_encoder = CrossEncoder(MODELO_CROSS_ENCODER, device=dispositivo) if MODELO_CROSS_ENCODER else None

print("[3/3] LLM...")
if BACKEND_LLM == "transformers":
    import transformers
    from transformers import pipeline
    from packaging import version
    arg_dtype = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56") else "torch_dtype"
    llm = pipeline("text-generation", model=MODELO_LLM, device=0 if dispositivo == "cuda" else -1,
                   **{arg_dtype: torch.float16 if dispositivo == "cuda" else torch.float32})
else:
    import ollama
    llm = None
    # Confirma que o modelo configurado está disponível no servidor local.
    resposta_lista = ollama.list()
    lista_modelos = getattr(resposta_lista, "models", resposta_lista.get("models", []) if isinstance(resposta_lista, dict) else [])
    modelos_instalados = [
        getattr(m, "model", m.get("name", "") if isinstance(m, dict) else str(m))
        for m in lista_modelos
    ]
    if not any(MODELO_OLLAMA in nome for nome in modelos_instalados):
        raise RuntimeError(
            f"Modelo {MODELO_OLLAMA} não encontrado. Execute a célula Ollama no final do notebook."
        )

def gerar_llm(mensagens, max_tokens=350):
    """Interface única para a LLM, com geração determinística para respostas consistentes."""
    if BACKEND_LLM == "transformers":
        saida = llm(mensagens, max_new_tokens=max_tokens, do_sample=False)
        texto = saida[0]["generated_text"][-1]["content"]
    else:
        r = ollama.chat(
            model=MODELO_OLLAMA,
            messages=mensagens,
            options={
                "temperature": 0.1,
                "top_p": 0.85,
                "repeat_penalty": 1.1,
                "num_ctx": 8192,
                "num_predict": max_tokens
            }
        )
        mensagem = getattr(r, "message", None)
        texto = mensagem.content if mensagem is not None else r["message"]["content"]

    # Remove o bloco de raciocínio interno do DeepSeek-R1, quando presente.
    texto = re.sub(r"<think>.*?</think>", "", texto, flags=re.S)
    return texto.split("</think>")[-1].strip()

# Banco vetorial. O nome da coleção inclui o modelo de embedding, porque vetores de
# modelos diferentes têm tamanhos diferentes e NÃO podem ser misturados.
cliente = chromadb.PersistentClient(path=CAMINHO_BANCO)
NOME_COLECAO = "docs_" + re.sub(r"[^a-zA-Z0-9]+", "_", MODELO_BI_ENCODER.split("/")[-1])[:50]
colecao = cliente.get_or_create_collection(
    name=NOME_COLECAO, metadata={"hnsw:space": "cosine", "hnsw:M": 32, "hnsw:construction_ef": 100})

TABELAS = {}
for arq in sorted(os.listdir(PASTA_TABELAS)):
    if arq.endswith(".pkl"):
        df_salvo = pd.read_pickle(os.path.join(PASTA_TABELAS, arq))
        TABELAS[df_salvo.attrs.get("nome_original", arq[:-4])] = df_salvo

print(f"\n✔ Pronto. Coleção '{NOME_COLECAO}' com {colecao.count()} fragmentos | "
      f"tabelas salvas: {list(TABELAS) or 'nenhuma'}")

[1/3] Bi-encoder...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[2/3] Cross-encoder...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[3/3] LLM...

✔ Pronto. Coleção 'docs_paraphrase_multilingual_MiniLM_L12_v2' com 0 fragmentos | tabelas salvas: nenhuma


## Célula 4 — Leitura dos arquivos e ingestão
- **Textos** (PDF, DOCX, DOC, TXT) são divididos em fragmentos por frases e viram vetores no ChromaDB.
- **Tabelas** (CSV, TSV, XLSX, ou TXT com colunas) são lidas como DataFrame. A leitura reconhece o padrão brasileiro: separador `;`, números como `1.234,56` e `R$ 10,50`, datas `dd/mm/aaaa` e acentos do Excel. A tabela completa é salva em disco para os cálculos, e as linhas também viram fragmentos no ChromaDB para a busca semântica.
- Reenviar um arquivo com o mesmo nome **substitui** a versão anterior.

In [16]:
# ==============================================================================
# CÉLULA 4 — LEITURA DE ARQUIVOS E INGESTÃO (PDF, DOCX, DOC, TXT, CSV, TSV, XLSX)
# ------------------------------------------------------------------------------
# • Textos (PDF/DOC/DOCX/TXT) → fragmentos → vetores no ChromaDB
# • Tabelas (CSV/TSV/XLSX/TXT tabular) → DataFrame salvo em disco (para estatística exata)
#   E TAMBÉM fragmentos com linhas inteiras no ChromaDB (para busca semântica)
# ==============================================================================

def normalizar(txt):
    """minúsculas, sem acentos, sem _ e -: 'Preço_Unitário' -> 'preco unitario'"""
    txt = unicodedata.normalize("NFKD", str(txt)).encode("ascii", "ignore").decode()
    txt = re.sub(r"[^a-z0-9.,%]+", " ", txt.lower())          # pontuação vira espaço
    txt = re.sub(r"(?<!\d)[.,]|[.,](?!\d)", " ", txt)        # mantém só '1.234,5'
    return re.sub(r"\s+", " ", txt).strip()

def decodificar(conteudo):
    """Arquivos do Excel brasileiro costumam vir em Windows-1252, não em UTF-8."""
    for enc in ("utf-8-sig", "cp1252", "latin-1"):
        try:
            return conteudo.decode(enc)
        except UnicodeDecodeError:
            continue
    return conteudo.decode("utf-8", errors="ignore")

def detectar_separador(texto):
    amostra = "\n".join(texto.splitlines()[:30])
    try:
        return csv.Sniffer().sniff(amostra, delimiters=";,\t|").delimiter
    except csv.Error:
        contagem = {s: amostra.count(s) for s in [";", ",", "\t", "|"]}
        return max(contagem, key=contagem.get)

def parece_tabela(texto):
    """Um .txt com colunas separadas por ; , TAB ou | é tratado como tabela."""
    linhas = [l for l in texto.splitlines() if l.strip()][:15]
    if len(linhas) < 3:
        return False
    sep = detectar_separador(texto)
    qtd = [l.count(sep) for l in linhas]
    return qtd[0] >= 1 and all(q == qtd[0] for q in qtd)

# ---------- conversão de números no padrão brasileiro (1.234,56 / R$ 10,50 / 15%) ----------
_RE_MILHAR_BR = re.compile(r"^-?\d{1,3}(\.\d{3})+(,\d+)?$")
_RE_MILHAR_US = re.compile(r"^-?\d{1,3}(,\d{3})+(\.\d+)?$")

def _para_numero(valor, padrao_br):
    s = str(valor).strip()
    if s == "" or s.lower() in ("nan", "none", "null", "-", "n/a"):
        return None                                   # célula vazia
    s = re.sub(r"(?i)r\$|us\$|€|%", "", s).replace("\u00a0", "").replace(" ", "")
    negativo = s.startswith("(") and s.endswith(")")
    s = s.strip("()")
    if "," in s and "." in s:
        s = s.replace(".", "").replace(",", ".") if s.rfind(",") > s.rfind(".") else s.replace(",", "")
    elif "," in s:
        s = s.replace(",", "") if (not padrao_br and _RE_MILHAR_US.match(s)) else s.replace(",", ".")
    elif padrao_br and _RE_MILHAR_BR.match(s):
        s = s.replace(".", "")
    try:
        n = float(s)
        return -n if negativo else n
    except ValueError:
        return float("nan")                           # texto que não é número

def tentar_numerico(serie, padrao_br):
    convertidos, validos, falhas = [], 0, 0
    for v in serie:
        r = _para_numero(v, padrao_br)
        if r is None:
            convertidos.append(np.nan)
        elif r != r:                                  # NaN = falhou
            falhas += 1
            convertidos.append(np.nan)
        else:
            validos += 1
            convertidos.append(r)
    if validos and falhas <= 0.1 * (validos + falhas):
        return pd.Series(convertidos, index=serie.index, dtype="float64")
    return None

_RE_DATA = re.compile(r"^(\d{1,2}[/.\-]\d{1,2}[/.\-]\d{2,4}|\d{4}-\d{2}-\d{2})")

def tentar_data(serie):
    nn = serie.dropna().astype(str).str.strip()
    nn = nn[nn != ""]
    if len(nn) == 0 or nn.str.match(_RE_DATA).mean() < 0.8:
        return None
    iso = nn.str.match(r"^\d{4}-").mean() > 0.5
    try:
        d = pd.to_datetime(serie, errors="coerce", dayfirst=not iso, format="mixed")
    except (TypeError, ValueError):
        d = pd.to_datetime(serie, errors="coerce", dayfirst=not iso)
    return d if d.notna().sum() >= 0.8 * len(nn) else None

def converter_tipos(df, padrao_br):
    df = df.copy()
    for c in df.columns:
        datas = tentar_data(df[c])
        if datas is not None:
            df[c] = datas
            continue
        numeros = tentar_numerico(df[c], padrao_br)
        if numeros is not None:
            df[c] = numeros
        else:
            df[c] = df[c].astype("string").str.strip()
    return df

def ler_tabela(conteudo, nome):
    ext = nome.rsplit(".", 1)[-1].lower()
    if ext in ("xlsx", "xls"):
        df = pd.read_excel(io.BytesIO(conteudo), dtype=str)
        padrao_br = True
    else:
        texto = decodificar(conteudo)
        sep = "\t" if ext == "tsv" else detectar_separador(texto)
        df = pd.read_csv(io.StringIO(texto), sep=sep, dtype=str, skipinitialspace=True)
        padrao_br = sep != ","            # ';' quase sempre indica CSV do Excel em pt-BR
    df.columns = [str(c).strip() for c in df.columns]
    df = df.loc[:, ~df.columns.str.match(r"^Unnamed")].dropna(how="all")
    df = df.reset_index(drop=True)
    df = converter_tipos(df, padrao_br)
    df.attrs["nome_original"] = nome
    return df

def tipo_coluna(serie):
    if pd.api.types.is_datetime64_any_dtype(serie):
        return "data"
    if pd.api.types.is_numeric_dtype(serie):
        return "número"
    return "texto"

def fmt_num(x, casas=2):
    """Formata no padrão brasileiro: 1234.5 -> '1.234,50'"""
    if x is None or pd.isna(x):
        return "—"
    x = float(x)
    s = f"{int(round(x)):,}" if x.is_integer() and abs(x) < 1e15 else f"{x:,.{casas}f}"
    return s.replace(",", "X").replace(".", ",").replace("X", ".")

def formatar_valor(v):
    if isinstance(v, pd.Timestamp):
        return v.strftime("%d/%m/%Y")
    if isinstance(v, (int, float, np.number)) and not isinstance(v, bool):
        return fmt_num(v)
    return str(v)

def extrair_texto(conteudo, nome):
    ext = nome.rsplit(".", 1)[-1].lower()
    if ext == "pdf":
        import pypdf
        leitor = pypdf.PdfReader(io.BytesIO(conteudo))
        return "\n".join((p.extract_text() or "") for p in leitor.pages).strip()
    if ext == "doc":                       # Word antigo: convertido pelo programa antiword
        with tempfile.NamedTemporaryFile(suffix=".doc", delete=False) as tmp:
            tmp.write(conteudo)
        r = subprocess.run(["antiword", tmp.name], capture_output=True, text=True)
        os.remove(tmp.name)
        if r.returncode != 0:
            raise ValueError("não foi possível ler o .doc (salve-o como .docx e envie de novo)")
        return r.stdout.strip()
    if ext == "docx":
        import docx
        documento = docx.Document(io.BytesIO(conteudo))
        return "\n".join(p.text for p in documento.paragraphs if p.text.strip()).strip()
    return decodificar(conteudo).strip()

def criar_chunks(texto, tamanho=600):
    """Corta por frases (não no meio de palavras) e repete a última frase como sobreposição."""
    frases = [f.strip() for f in re.split(r"(?<=[.!?;])\s+|\n{2,}", texto) if f.strip()]
    chunks, atual = [], []
    for frase in frases:
        while len(frase) > tamanho:                          # frase gigante: corta por palavras
            corte = frase.rfind(" ", 0, tamanho)
            corte = corte if corte > 0 else tamanho
            chunks.append(frase[:corte])
            frase = frase[corte:].strip()
        if atual and len(" ".join(atual + [frase])) > tamanho:
            chunks.append(" ".join(atual))
            atual = [atual[-1]]                              # sobreposição de 1 frase
        atual.append(frase)
    if atual:
        chunks.append(" ".join(atual))
    return chunks

def tabela_para_chunks(df, nome, linhas_por_chunk=5, max_linhas=3000):
    """Cada fragmento leva o nome do arquivo e linhas INTEIRAS (nunca corta um registro ao meio)."""
    colunas = ", ".join(f"{c} ({tipo_coluna(df[c])})" for c in df.columns)
    chunks = [f"Tabela '{nome}' com {len(df)} linhas e {len(df.columns)} colunas: {colunas}."]
    for ini in range(0, min(len(df), max_linhas), linhas_por_chunk):
        linhas = []
        for n, (_, row) in enumerate(df.iloc[ini:ini + linhas_por_chunk].iterrows(), start=ini + 1):
            campos = "; ".join(f"{c}: {formatar_valor(v)}" for c, v in row.items() if pd.notna(v))
            linhas.append(f"Linha {n}: {campos}")
        chunks.append(f"Tabela '{nome}'\n" + "\n".join(linhas))
    return chunks

EXTENSOES_TABELA = ("csv", "tsv", "xlsx", "xls")

def ingerir_arquivo(nome, conteudo, setor="geral"):
    conteudo = bytes(conteudo)           # o FileUpload pode entregar memoryview
    ext = nome.rsplit(".", 1)[-1].lower()
    df = None
    if ext in EXTENSOES_TABELA or (ext == "txt" and parece_tabela(decodificar(conteudo))):
        df = ler_tabela(conteudo, nome)
        TABELAS[nome] = df
        nome_seguro = re.sub(r"[^\w.]+", "_", nome)
        df.to_pickle(os.path.join(PASTA_TABELAS, nome_seguro + ".pkl"))
        chunks, tipo = tabela_para_chunks(df, nome), "tabela"
    else:
        texto = extrair_texto(conteudo, nome)
        if not texto:
            raise ValueError("nenhum texto extraído (PDF escaneado?)")
        chunks, tipo = criar_chunks(texto), "texto"

    # Reenviar o mesmo arquivo substitui a versão antiga, em vez de duplicar
    try:
        colecao.delete(where={"arquivo_origem": nome})
    except Exception:
        pass

    assinatura = hashlib.md5(nome.encode() + conteudo).hexdigest()[:10]
    ids = [f"{assinatura}_{i}" for i in range(len(chunks))]
    metas = [{"arquivo_origem": nome, "setor": setor, "tipo": tipo,
              "tema": f"Anexo: {nome}", "ref_url": f"local://{nome}"} for _ in chunks]
    vetores = bi_encoder.encode([PREFIXO_DOCUMENTO + c for c in chunks],
                                normalize_embeddings=True, batch_size=64).tolist()
    for i in range(0, len(chunks), 500):
        colecao.upsert(ids=ids[i:i+500], documents=chunks[i:i+500],
                       embeddings=vetores[i:i+500], metadatas=metas[i:i+500])
    return {"nome": nome, "tipo": tipo, "fragmentos": len(chunks), "df": df}


def ingerir_caminho(caminho, setor="geral"):
    """O Gradio entrega o CAMINHO do arquivo enviado; lemos os bytes e ingerimos."""
    nome = os.path.basename(caminho)
    with open(caminho, "rb") as f:
        return ingerir_arquivo(nome, f.read(), setor)

## Célula 5 — Motor estatístico
Interpreta perguntas como *"total de X por Y"*, *"média de X na região Sul"*, *"gráfico da evolução mensal"* ou *"mostre a linha 3"* e calcula com **pandas**.

> ⚠️ **Por que não deixar a LLM calcular?** A busca vetorial entrega só alguns trechos, e uma LLM pequena erra contas. Para somar 120 linhas, o caminho certo é o pandas calcular o número exato e a LLM apenas explicar o resultado.

In [17]:
# ==============================================================================
# CÉLULA 5 — MOTOR ANALÍTICO: média, total, máx/mín, contagem, agrupamento, gráfico
# A LLM NÃO faz conta: o pandas calcula o valor exato e a LLM só explica o resultado.
# ==============================================================================

OPERACOES = [  # (nome, regex sobre a pergunta normalizada, função pandas, rótulo)
    ("mediana",  r"\bmedianas?\b",                                             "median", "Mediana"),
    ("media",    r"\b(medi[ao]s?|em media)\b",                                 "mean",   "Média"),
    ("soma",     r"\b(total|totais|totalizador|totalizar|soma|somar|somatorio|somad[oa]s?|acumulad[oa])\b", "sum", "Total"),
    ("max",      r"\b(maxim[oa]s?|maior(es)?|mais alt[oa]s?|pico)\b",          "max",    "Máximo"),
    ("min",      r"\b(minim[oa]s?|menor(es)?|mais baix[oa]s?)\b",              "min",    "Mínimo"),
    ("contagem", r"\b(quant[oa]s|quantidade de|contagem|contar|numero de|qtd)\b", "count", "Contagem"),
    ("desvio",   r"\bdesvio( padrao)?\b",                                      "std",    "Desvio-padrão"),
]
FUNCAO = {nome: (f, rot) for nome, _, f, rot in OPERACOES}
RE_RESUMO  = r"\b(resumo|estatisticas?|descreva|visao geral|describe)\b"
RE_GRAFICO = r"\b(grafic\w*|plot\w*|visualiz\w*|chart|barras?|evolucao|tendencia)\b"
RE_TABELA  = r"\b(tabela|listar|liste|lista|mostre|mostrar|exiba|exibir|registros)\b"
RE_LINHA   = r"\b(?:linha|registro) (\d+)\b"
RE_DADOS   = r"\b(registros|linhas|tabela|planilha|csv|dados|base)\b"
PERIODOS = {"dia": "D", "diario": "D", "semana": "W", "semanal": "W", "mes": "M", "meses": "M",
            "mensal": "M", "trimestre": "Q", "trimestral": "Q", "ano": "Y", "anos": "Y", "anual": "Y"}
COMPARADORES = [
    (r"(?:acima de|maior(?:es)? que|mais de|superior(?:es)? a|>=?)\s*([\d.,]+)", ">"),
    (r"(?:abaixo de|menor(?:es)? que|menos de|inferior(?:es)? a|<=?)\s*([\d.,]+)", "<"),
]

PALAVRAS_OPERACAO = {"total", "soma", "media", "medio", "maximo", "minimo", "maior", "menor",
                     "contagem", "numero", "geral", "dado"}
RE_VALOR_PREFERIDO = r"\b(total|valor|receita|faturamento|venda|montante|custo|lucro)\b"
RE_NAO_SOMAR = r"\b(id|cod|codigo|cep|preco|unitario|taxa|percentual|aliquota|ano|mes|dia)\b"

def _singular(p):
    return p[:-1] if len(p) > 3 and p.endswith("s") else p

def _texto_busca(q):
    return " " + " ".join(_singular(p) for p in q.split()) + " "

def colunas_citadas(q, df):
    """Encontra colunas mencionadas na pergunta (tolera plural, acento e pequenos erros)."""
    tq = _texto_busca(q)
    palavras = tq.split()
    achadas = []
    for c in df.columns:
        tokens = [_singular(t) for t in normalizar(c).split()]
        if not tokens:
            continue
        alvo = " " + " ".join(tokens) + " "
        if alvo in tq:
            achadas.append((tq.find(alvo), c, alvo))
        elif len(tokens) > 1 and all(t in palavras for t in tokens if len(t) > 2):
            achadas.append((tq.find(" " + tokens[0] + " "), c, None))
        elif len(tokens) > 1 and any(t in palavras for t in tokens if len(t) >= 5 and t not in PALAVRAS_OPERACAO):
            t = next(t for t in tokens if len(t) >= 5 and t not in PALAVRAS_OPERACAO and t in palavras)
            achadas.append((tq.find(" " + t + " "), c, " " + t + " "))
        elif len(tokens) == 1 and len(tokens[0]) >= 5:
            for p in palavras:
                if len(p) >= 5 and difflib.SequenceMatcher(None, tokens[0], p).ratio() >= 0.88:
                    achadas.append((tq.find(" " + p + " "), c, " " + p + " "))
                    break
    achadas.sort()
    return [(c, alvo) for _, c, alvo in achadas]

def _valores_citados(q, df, ignorar):
    """Filtros por valor: 'região Sul' -> df[regiao == 'Sul']"""
    tq = _texto_busca(q)
    filtros = {}
    for c in df.columns:
        if c in ignorar or tipo_coluna(df[c]) != "texto" or df[c].nunique() > 500:
            continue
        for v in df[c].dropna().unique():
            nv = " " + " ".join(_singular(p) for p in normalizar(v).split()) + " "
            if len(nv.strip()) >= 2 and nv in tq:
                filtros.setdefault(c, []).append(v)
    return filtros

def escolher_tabela(pergunta, escolhida="auto"):
    if not TABELAS:
        return None, None
    if escolhida != "auto" and escolhida in TABELAS:
        return escolhida, TABELAS[escolhida]
    q = normalizar(pergunta)
    melhor, melhor_pontos = list(TABELAS)[-1], -1       # empate -> a mais recente
    for nome, df in TABELAS.items():
        pontos = 3 * len(colunas_citadas(q, df)) + len(_valores_citados(q, df, []))
        pontos += 5 if normalizar(nome.rsplit(".", 1)[0]) in q else 0
        if pontos >= melhor_pontos:
            melhor, melhor_pontos = nome, pontos
    return melhor, TABELAS[melhor]

def planejar(pergunta, df, tabela_forcada=False):
    """Traduz a pergunta em um plano de análise. Devolve None se não for pergunta analítica."""
    q = normalizar(pergunta)
    plano = {"ops": [], "valores": [], "grupo": None, "periodo": None, "filtros": {},
             "comparacoes": [], "grafico": bool(re.search(RE_GRAFICO, q)),
             "tabela": bool(re.search(RE_TABELA, q)), "linha": None, "resumo": bool(re.search(RE_RESUMO, q))}

    m = re.search(RE_LINHA, q)
    if m:
        plano["linha"] = int(m.group(1))
        return plano

    numericas = [c for c in df.columns if tipo_coluna(df[c]) == "número"]
    datas = [c for c in df.columns if tipo_coluna(df[c]) == "data"]

    # comparações numéricas ("vendas acima de 1000") — retiradas da frase antes de achar operações
    q_ops = q
    for regex, sinal in COMPARADORES:
        for mm in re.finditer(regex, q):
            plano["comparacoes"].append((sinal, _para_numero(mm.group(1), True)))
        q_ops = re.sub(regex, " ", q_ops)

    # agrupamento: "por região", "para cada produto", "por mês"
    mg = re.search(r"\b(?:por|cada|agrupad[oa]s? por|segundo)\s+(?:cada\s+|o\s+|a\s+)?(.+)", q)
    if mg:
        trecho = " ".join(mg.group(1).split()[:4])
        primeira = trecho.split()[0] if trecho else ""
        if primeira in PERIODOS and datas:
            plano["grupo"], plano["periodo"] = datas[0], PERIODOS[primeira]
        else:
            cand = colunas_citadas(trecho, df)
            if cand:
                plano["grupo"] = cand[0][0]
    if not plano["grupo"] and datas:
        for palavra, per in PERIODOS.items():
            if per in "MQY" and re.search(rf"\b{palavra}\b", q) and re.search(r"\b(mensal|anual|trimestral|evolucao|tendencia)\b", q):
                plano["grupo"], plano["periodo"] = datas[0], per
                break

    # colunas de valor citadas (escondidas da frase para não confundir 'quantidade' coluna x operação)
    citadas = colunas_citadas(q_ops, df)
    for c, alvo in citadas:
        if c in numericas and c != plano["grupo"]:
            plano["valores"].append(c)
        if alvo:
            q_ops = _texto_busca(q_ops).replace(alvo, " ")

    for nome, regex, _, _ in OPERACOES:
        if re.search(regex, q_ops):
            plano["ops"].append(nome)
    if not plano["ops"] and not plano["valores"]:            # ex.: 'quantidade por produto'
        plano["ops"] = [n for n, r, _, _ in OPERACOES if re.search(r, q)]
    if "soma" in plano["ops"] and not plano["valores"] and re.search(r"\b(registros|linhas|itens)\b", q):
        plano["ops"] = ["contagem"]
    if "media" in plano["ops"] and "mediana" in plano["ops"] and not re.search(r"\bmedi[ao]s?\b", q):
        plano["ops"].remove("media")

    # filtros por valor de texto e por ano
    plano["filtros"] = _valores_citados(q, df, ignorar=[plano["grupo"]])
    anos = re.findall(r"\b(19\d{2}|20\d{2})\b", q)
    if anos and datas:
        plano["filtros"][f"__ano__{datas[0]}"] = [int(a) for a in anos]

    # Só é analítica se a pergunta "ancorou" em algo concreto da tabela
    ancorada = bool(plano["valores"] or plano["grupo"] or plano["filtros"] or plano["comparacoes"])
    ancorada = ancorada or tabela_forcada or (bool(re.search(RE_DADOS, q)) and
                                              (plano["ops"] or plano["resumo"] or plano["grafico"] or plano["tabela"]))
    if not ancorada:
        return None
    if not (plano["ops"] or plano["resumo"] or plano["grafico"] or plano["tabela"]):
        if plano["grupo"]:
            plano["ops"] = ["soma"]
        elif plano["filtros"] or plano["comparacoes"] or plano["valores"]:
            plano["tabela"] = True
        else:
            return None
    return plano

def colunas_de_valor(df, plano):
    """Colunas citadas na pergunta; se nenhuma, escolhe as mais prováveis (valor, total, receita...)."""
    if plano["valores"]:
        return plano["valores"]
    numericas = [c for c in df.columns if tipo_coluna(df[c]) == "número"]
    if plano["resumo"] and not plano["ops"]:
        return numericas
    somaveis = [c for c in numericas if not re.search(RE_NAO_SOMAR, normalizar(c))]
    preferidas = [c for c in somaveis if re.search(RE_VALOR_PREFERIDO, normalizar(c))]
    return (preferidas or somaveis or numericas)[:1 if (plano["grupo"] or plano["grafico"]) else 3]

def _aplicar_filtros(df, plano):
    codigo, df_f = [], df
    for col, valores in plano["filtros"].items():
        if col.startswith("__ano__"):
            c = col[7:]
            df_f = df_f[df_f[c].dt.year.isin(valores)]
            codigo.append(f"df['{c}'].dt.year.isin({valores})")
        else:
            df_f = df_f[df_f[col].isin(valores)]
            codigo.append(f"df['{col}'].isin({list(map(str, valores))})")
    if plano["comparacoes"]:
        alvo = (colunas_de_valor(df, plano) or [None])[0]
        if alvo:
            for sinal, limite in plano["comparacoes"]:
                df_f = df_f[df_f[alvo] > limite] if sinal == ">" else df_f[df_f[alvo] < limite]
                codigo.append(f"df['{alvo}'] {sinal} {limite:g}")
    return df_f, codigo

def _chave_grupo(df_f, plano):
    if plano["periodo"]:
        return df_f[plano["grupo"]].dt.to_period(plano["periodo"]).astype(str), f"df['{plano['grupo']}'].dt.to_period('{plano['periodo']}')"
    return df_f[plano["grupo"]], f"'{plano['grupo']}'"

def gerar_grafico(resultado, titulo, eh_tempo):
    fig, ax = plt.subplots(figsize=(9, 4.6))
    dados = resultado if isinstance(resultado, pd.DataFrame) else resultado.to_frame()
    dados = dados.astype(float)
    if eh_tempo:
        dados.plot(ax=ax, marker="o", linewidth=2)
    elif len(dados) > 8:
        dados.iloc[::-1].plot.barh(ax=ax, width=0.75)
    else:
        dados.plot.bar(ax=ax, width=0.75, rot=0)
    if len(dados.columns) == 1:
        ax.get_legend().remove() if ax.get_legend() else None
        for barra in ax.containers:
            ax.bar_label(barra, labels=[fmt_num(v) for v in barra.datavalues], padding=3, fontsize=9)
    eixo = ax.xaxis if (len(dados) > 8 and not eh_tempo) else ax.yaxis
    eixo.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: fmt_num(v)))
    ax.set_title(titulo, fontsize=13, fontweight="bold")
    ax.set_xlabel(""); ax.set_ylabel("")
    ax.spines[["top", "right"]].set_visible(False)
    ax.margins(x=0.08, y=0.12)
    fig.tight_layout()
    return fig

def executar_analise(df, plano, nome_tabela):
    """Executa o plano com pandas e devolve cards, tabela, gráfico e o código equivalente."""
    out = {"tabela_nome": nome_tabela, "cards": [], "tabela": None, "fig": None, "codigo": [], "titulo": ""}

    if plano["linha"]:
        n = plano["linha"]
        if not 1 <= n <= len(df):
            out["titulo"] = f"A tabela tem {len(df)} linhas; a linha {n} não existe."
            return out
        out["tabela"] = df.iloc[[n - 1]].T.rename(columns={n - 1: f"Linha {n}"})
        out["titulo"] = f"Linha {n} de '{nome_tabela}'"
        out["codigo"].append(f"df.iloc[{n - 1}]")
        return out

    df_f, cod_filtros = _aplicar_filtros(df, plano)
    base = "df" if not cod_filtros else "df[" + " & ".join(f"({c})" for c in cod_filtros) + "]"
    out["codigo"].append(f"dados = {base}")
    partes_filtro = [f"{k.replace('__ano__', 'ano de ')} = {', '.join(map(str, v))}" for k, v in plano["filtros"].items()]
    partes_filtro += [f"{(colunas_de_valor(df, plano) or ['valor'])[0]} {s} {fmt_num(l)}" for s, l in plano["comparacoes"]]
    filtros_txt = "; ".join(partes_filtro)
    out["cards"].append(("Registros analisados", fmt_num(len(df_f))))
    if len(df_f) == 0:
        out["titulo"] = "Nenhum registro atende aos filtros."
        return out

    numericas = [c for c in df.columns if tipo_coluna(df[c]) == "número"]
    ops = plano["ops"]
    valores = colunas_de_valor(df, plano)

    if plano["resumo"] and not ops:
        out["tabela"] = df_f[valores].describe().T.rename(columns={
            "count": "contagem", "mean": "média", "std": "desvio", "min": "mínimo", "max": "máximo"})
        out["titulo"] = f"Resumo estatístico de '{nome_tabela}'"
        out["codigo"].append(f"dados[{valores}].describe()")
        for c in valores[:4]:
            out["cards"] += [(f"Total de {c}", fmt_num(df_f[c].sum())), (f"Média de {c}", fmt_num(df_f[c].mean()))]
        return out

    # gráfico sem operação nem grupo: escolhe um agrupamento sensato
    if plano["grafico"] and not plano["grupo"]:
        categoricas = [c for c in df.columns if tipo_coluna(df[c]) == "texto" and 1 < df_f[c].nunique() <= 30]
        datas = [c for c in df.columns if tipo_coluna(df[c]) == "data"]
        if datas:
            plano["grupo"], plano["periodo"] = datas[0], "M"
        elif categoricas:
            plano["grupo"] = categoricas[0]
    if not ops and (plano["grupo"] or plano["grafico"]):
        ops = ["soma"]

    if plano["grupo"]:
        chave, cod_chave = _chave_grupo(df_f, plano)
        rotulo_periodo = {"D": "dia", "W": "semana", "M": "mês", "Q": "trimestre", "Y": "ano"}
        nome_grupo = plano["grupo"] + (f" ({rotulo_periodo[plano['periodo']]})" if plano["periodo"] else "")
        partes = {}
        for op in ops or ["soma"]:
            f, rot = FUNCAO[op]
            if op == "contagem":
                partes["Contagem"] = df_f.groupby(chave).size()
                out["codigo"].append(f"dados.groupby({cod_chave}).size()")
            else:
                for c in valores[:3]:
                    partes[f"{rot} de {c}"] = df_f.groupby(chave)[c].agg(f)
                out["codigo"].append(f"dados.groupby({cod_chave})[{valores[:3]}].{f}()")
        resultado = pd.DataFrame(partes)
        resultado.index.name = nome_grupo
        if not plano["periodo"]:
            resultado = resultado.sort_values(resultado.columns[0], ascending=("min" in ops))
        out["tabela"] = resultado.head(50)
        out["titulo"] = f"{', '.join(resultado.columns)} por {nome_grupo}" + (f" — {filtros_txt}" if filtros_txt else "")
        primeira = resultado.columns[0]
        destaque = resultado[primeira].idxmin() if "min" in ops else resultado[primeira].idxmax()
        out["cards"].append((f"Destaque ({primeira})", f"{destaque}: {fmt_num(resultado.loc[destaque, primeira])}"))
        if any(op in ("soma", "contagem") for op in ops):
            out["cards"].append((f"Soma geral ({primeira})", fmt_num(resultado[primeira].sum())))
        if plano["grafico"] or len(resultado) > 1:
            out["fig"] = gerar_grafico(resultado.head(20).iloc[:, :3], out["titulo"], bool(plano["periodo"]))
        return out

    # sem agrupamento: totalizadores (cards)
    for op in ops:
        f, rot = FUNCAO[op]
        if op == "contagem":
            out["cards"].append(("Contagem de registros", fmt_num(len(df_f))))
            continue
        for c in valores:
            out["cards"].append((f"{rot} de {c}", fmt_num(df_f[c].agg(f))))
            if op in ("max", "min"):
                linha = df_f.loc[df_f[c].idxmax() if op == "max" else df_f[c].idxmin()]
                out["tabela"] = linha.to_frame(f"Registro com {rot.lower()} {c}")
        out["codigo"].append(f"dados[{valores}].{f}()")
    if plano["tabela"] and out["tabela"] is None:
        out["tabela"] = df_f.head(50)
        out["codigo"].append("dados.head(50)")
    out["titulo"] = ("Resultado" if ops else "Registros encontrados") + (f" — {filtros_txt}" if filtros_txt else "")
    return out

## Célula 6 — RAG em duas etapas e roteador
O **roteador** decide o caminho de cada pergunta. Se ela cita colunas, valores ou operações de uma tabela, vai para o motor estatístico. Caso contrário, segue o RAG: bi-encoder (top-10) → cross-encoder → filtro de relevância (≥ 60% da nota do 1º) → LLM com prompt rígido.

In [18]:
# ==============================================================================
# CÉLULA 6 — RAG EM DUAS ETAPAS + ROTEADOR (texto × estatística)
# ------------------------------------------------------------------------------
#   pergunta ──► é pergunta estatística sobre uma tabela? ──sim──► pandas (cálculo exato)
#                        │ não                                      + gráfico + LLM explica
#                        ▼
#            Etapa 1: bi-encoder + HNSW (top-10)
#            Etapa 2: cross-encoder (re-ranking) + filtro de relevância
#            Geração: LLM responde só com os trechos selecionados
# ==============================================================================

SETORES = [("Geral", "geral"), ("Fisiologia cardiovascular", "fisiologia"),
           ("Desempenho esportivo", "desempenho"), ("Arquitetura e dados (IoT/AWS)", "arquitetura"),
           ("Segurança e restrições", "seguranca")]

# Documentos de exemplo do projeto Haja Coração (resumidos da documentação do Grupo 4)
ARTIGOS_EXEMPLO = [
    # ---------------- FISIOLOGIA CARDIOVASCULAR ----------------
    {
        "id": "doc_fisiologia_01",
        "setor": "fisiologia",
        "tema": "Fisiologia (Esforço cardiovascular no futebol)",
        "texto": (
            "O futebol profissional exige grande esforço cardiovascular. Durante um jogo, os atletas fazem sprints, "
            "mudanças rápidas de direção e disputas físicas, e a carga emocional também eleva a frequência cardíaca. "
            "Em esforço máximo, os jogadores podem chegar perto do seu limite fisiológico de batimentos por minuto (BPM), "
            "e o organismo precisa se recuperar rápido para manter o desempenho até o fim da partida."
        ),
        "ref": "local://documentacao_grupo4#contexto",
    },
    {
        "id": "doc_fisiologia_02",
        "setor": "fisiologia",
        "tema": "Fisiologia (Limites dos exames periódicos)",
        "texto": (
            "O acompanhamento cardiovascular dos atletas ainda depende de avaliações periódicas, como exames clínicos, "
            "eletrocardiogramas e testes ergométricos. Esses exames não mostram o comportamento do atleta em esforço real, "
            "como treinos intensos e partidas. Há casos de atletas com exames normais que só apresentaram alterações "
            "em esforço elevado, como arritmias leves, fadiga cardíaca ou recuperação inadequada."
        ),
        "ref": "local://documentacao_grupo4#contexto",
    },
    # ---------------- DESEMPENHO ESPORTIVO ----------------
    {
        "id": "doc_desempenho_01",
        "setor": "desempenho",
        "tema": "Desempenho (Carga de treino e overtraining)",
        "texto": (
            "Quando o atleta é submetido a esforço elevado sem tempo adequado de recuperação, pode ocorrer sobrecarga "
            "física, conhecida como overtraining, que causa queda de desempenho, fadiga persistente e maior risco de lesões. "
            "Por outro lado, monitorar os níveis fisiológicos permite potencializar os treinos e melhorar o resultado."
        ),
        "ref": "local://documentacao_grupo4#contexto",
    },
    {
        "id": "doc_desempenho_02",
        "setor": "desempenho",
        "tema": "Desempenho (Objetivos do monitoramento)",
        "texto": (
            "Objetivos do projeto: implementar sensores vestíveis para coletar dados cardiovasculares; monitorar a frequência "
            "cardíaca por minuto durante treinos e partidas; registrar métricas de esforço físico e recuperação; identificar "
            "padrões fisiológicos; detectar variações anormais ligadas ao esforço; e disponibilizar os dados em dashboards."
        ),
        "ref": "local://documentacao_grupo4#objetivos",
    },
    # ---------------- ARQUITETURA E DADOS ----------------
    {
        "id": "doc_arquitetura_01",
        "setor": "arquitetura",
        "tema": "Arquitetura (Tecnologias do sistema)",
        "texto": (
            "Tecnologias do sistema: Python para coleta, processamento e análise de dados; Java para serviços, endpoints "
            "HTTP/HTTPS e integração; SQL para consultas; JavaScript para a interface web e os dashboards. Os dados ficam "
            "em um banco relacional MySQL normalizado, o processamento distribuído usa Spark e a infraestrutura é a nuvem AWS."
        ),
        "ref": "local://documentacao_grupo4#escopo",
    },
    {
        "id": "doc_arquitetura_02",
        "setor": "arquitetura",
        "tema": "Arquitetura (Pipeline de dados simulados)",
        "texto": (
            "Na versão atual, um script Python simula 11 sensores vestíveis (wearable-01 a wearable-11) e gera um arquivo "
            "JSON a cada 5 segundos com a frequência cardíaca, temperatura, latitude, longitude e posição em campo de cada "
            "jogador. Os arquivos vão para um bucket S3 e um consumidor grava os dados em um grafo Neo4j, em que o peso de "
            "cada conexão tática é a média da frequência cardíaca dos dois jogadores."
        ),
        "ref": "local://neo4j-graph-v0",
    },
    # ---------------- SEGURANÇA E RESTRIÇÕES ----------------
    {
        "id": "doc_seguranca_01",
        "setor": "seguranca",
        "tema": "Segurança (Boas práticas adotadas)",
        "texto": (
            "O projeto segue o OWASP Top 10 e as normas ISO/IEC 27000 e ISO/IEC 27001. As práticas previstas são: "
            "autenticação de usuários para controle de acesso, proteção dos dados sensíveis no banco, conexões seguras "
            "entre os componentes e desenvolvimento seguro."
        ),
        "ref": "local://documentacao_grupo4#seguranca",
    },
    {
        "id": "doc_seguranca_02",
        "setor": "seguranca",
        "tema": "Restrições (Uso acadêmico, sem diagnóstico)",
        "texto": (
            "O sistema é acadêmico: não tem certificação como dispositivo médico, não realiza diagnóstico real, não substitui "
            "exames clínicos e não toma decisões clínicas sem validação humana. Os atletas precisam autorizar a coleta dos "
            "dados fisiológicos, que serão usados apenas para fins acadêmicos e de pesquisa."
        ),
        "ref": "local://documentacao_grupo4#restricoes",
    },
]

def carregar_artigos_exemplo():
    """Grava os 8 documentos de exemplo do projeto (útil para testar a parte de texto)."""
    textos = [d["texto"] for d in ARTIGOS_EXEMPLO]
    vetores = bi_encoder.encode([PREFIXO_DOCUMENTO + t for t in textos], normalize_embeddings=True).tolist()
    colecao.upsert(ids=[d["id"] for d in ARTIGOS_EXEMPLO], documents=textos, embeddings=vetores,
                   metadatas=[{"arquivo_origem": d["tema"], "setor": d["setor"], "tipo": "texto",
                               "tema": d["tema"], "ref_url": d["ref"]} for d in ARTIGOS_EXEMPLO])
    return len(textos)


def para_probabilidade(scores):
    """Alguns cross-encoders devolvem logits (−∞..+∞), outros já devolvem 0..1."""
    scores = np.asarray(scores, dtype=float)
    if scores.min() >= 0 and scores.max() <= 1:
        return scores
    return 1 / (1 + np.exp(-scores))


def recuperar(pergunta, setor="todos"):
    """Etapas 1 e 2 do two-stage retrieval. Devolve (todos os candidatos, selecionados)."""
    total = colecao.count()
    if total == 0:
        return [], []
    v = bi_encoder.encode([PREFIXO_CONSULTA + pergunta], normalize_embeddings=True).tolist()
    filtro = {"where": {"setor": setor}} if setor != "todos" else {}
    busca = colecao.query(query_embeddings=v, n_results=min(K_CANDIDATOS, total), **filtro)
    candidatos = [{"doc": d, "meta": m, "sim": 1 - dist, "pos_bi": i}
                  for i, (d, m, dist) in enumerate(zip(busca["documents"][0], busca["metadatas"][0],
                                                       busca["distances"][0]), start=1)]
    if not candidatos:
        return [], []
    if cross_encoder is not None:
        notas = para_probabilidade(cross_encoder.predict([[pergunta, c["doc"]] for c in candidatos]))
        for c, n in zip(candidatos, notas):
            c["nota"] = float(n)
    else:                                         # sem re-ranking: a nota é o próprio cosseno
        for c in candidatos:
            c["nota"] = c["sim"]
    candidatos.sort(key=lambda c: c["nota"], reverse=True)
    topo = candidatos[0]
    # Não basta um documento ser o melhor entre os candidatos: ele também precisa
    # superar um mínimo de similaridade e de relevância configurável.
    if topo["sim"] < MIN_SIMILARIDADE or topo["nota"] < MIN_RELEVANCIA:
        return candidatos, []
    selecionados = [
        c for c in candidatos[:K_CONTEXTO]
        if c["nota"] >= max(MIN_RELEVANCIA, LIMIAR_RELATIVO * topo["nota"])
        and c["sim"] >= MIN_SIMILARIDADE
    ]
    return candidatos, selecionados


def responder_rag(pergunta, setor="todos"):
    candidatos, selecionados = recuperar(pergunta, setor)
    if not candidatos:
        return " Não encontrei documentos na base para esse filtro. Confira a aba **Base de conhecimento** ou envie documentos na aba **Documentos**.", None
    if not selecionados:
        return (
            "### Não encontrei evidências suficientes\n"
            "Os documentos recuperados não atingiram o nível mínimo de similaridade/relevância. "
            "Reformule a pergunta ou adicione materiais mais específicos à base. "
            "Não vou gerar uma resposta sem sustentação documental."
        ), None

    contexto = "\n\n".join(
        f"[FONTE_{i}] Documento: {c['meta'].get('arquivo_origem', 'Documento sem nome')}\n"
        f"Setor: {c['meta'].get('setor', 'não informado')}\n"
        f"Conteúdo: {c['doc']}"
        for i, c in enumerate(selecionados, start=1)
    )
    mensagens = [
        {"role": "system", "content": (
            "Você é o Haja Coração, assistente de apoio ao monitoramento cardiovascular e ao desempenho "
            "esportivo de atletas. Responda em português do Brasil, com clareza, objetividade e linguagem "
            "técnica acessível.\n\n"
            "REGRAS OBRIGATÓRIAS:\n"
            "1. Use exclusivamente as informações presentes nas fontes fornecidas.\n"
            "2. Não invente números, limites clínicos, causas, diagnósticos ou características de atletas.\n"
            "3. Diferencie fatos documentados de interpretações; não apresente inferências como fatos.\n"
            "4. Se a informação não estiver nas fontes, diga claramente: "
            "'Não encontrei essa informação nos documentos disponíveis.'\n"
            "5. Ao final de cada afirmação factual, indique a fonte correspondente, como [FONTE_1]. "
            "Não crie identificadores de fontes.\n"
            "6. Se a pergunta envolver saúde, deixe claro que o sistema é acadêmico e não substitui "
            "avaliação profissional ou diagnóstico médico. Só mencione isso quando for pertinente.\n"
            "7. Não misture informações de setores diferentes se elas não forem necessárias para responder."
        )},
        {"role": "user", "content": (
            f"FONTES RECUPERADAS:\n{contexto}\n\n"
            f"PERGUNTA DO USUÁRIO:\n{pergunta}\n\n"
            "Responda diretamente. Se possível, organize em tópicos curtos e cite as fontes."
        )},
    ]
    resposta = gerar_llm(mensagens, max_tokens=450)
    fontes = pd.DataFrame([{
        "Fonte": f"[FONTE_{i}]",
        "Documento": c["meta"].get("arquivo_origem", "doc"),
        "Setor": c["meta"].get("setor", "não informado"),
        "Similaridade": f"{c['sim']:.3f}",
        "Relevância": f"{c['nota'] * 100:.1f}%",
        "Trecho utilizado": c["doc"][:220] + ("…" if len(c["doc"]) > 220 else ""),
    } for i, c in enumerate(selecionados, start=1)])
    return f"### Resposta do Haja Coração\n\n{resposta}", fontes


def sugerir_perguntas(df):
    """Gera perguntas de exemplo a partir das colunas detectadas na tabela."""
    num = [c for c in df.columns if tipo_coluna(df[c]) == "número" and not re.search(RE_NAO_SOMAR, normalizar(c))]
    num.sort(key=lambda c: not re.search(RE_VALOR_PREFERIDO, normalizar(c)))   # 'valor', 'total'... primeiro
    cat = [c for c in df.columns if tipo_coluna(df[c]) == "texto" and 1 < df[c].nunique() <= 30]
    dat = [c for c in df.columns if tipo_coluna(df[c]) == "data"]
    s = []
    if num and cat:
        s += [f"Qual o total de {num[0]} por {cat[0]}? Faça um gráfico", f"Qual a média de {num[0]} por {cat[0]}?"]
    if num:
        s += [f"Qual o valor máximo de {num[0]}?"]
    if num and dat:
        s += [f"Gráfico da evolução mensal de {num[0]}"]
    return s + ["Faça um resumo estatístico dos dados", "Mostre a linha 3"]


def df_para_exibir(df, max_linhas=200):
    """Formata números no padrão brasileiro e transforma o índice em coluna (para o Gradio)."""
    d = df.head(max_linhas).copy()
    if not isinstance(d.index, pd.RangeIndex):
        d.index.name = d.index.name or "item"
        d = d.reset_index()
    fmt = lambda v: "—" if (v is None or (not isinstance(v, str) and pd.isna(v))) else formatar_valor(v)
    d = d.map(fmt) if hasattr(d, "map") else d.applymap(fmt)
    d.columns = [str(c) for c in d.columns]
    return d


def responder(pergunta, tabela_alvo="auto", setor="todos", explicar=True):
    """Roteador: decide entre análise estatística (pandas) e RAG em texto."""
    nome, df = escolher_tabela(pergunta, tabela_alvo)
    plano = planejar(pergunta, df, tabela_forcada=(tabela_alvo != "auto")) if df is not None else None
    if not plano:
        texto, fontes = responder_rag(pergunta, setor)
        return {"texto": texto, "fig": None, "tabela": fontes, "codigo": "", "rotulo": "Trechos recuperados"}

    out = executar_analise(df, plano, nome)
    linhas = "\n".join(f"| {r} | **{v}** |" for r, v in out["cards"])
    texto = (f"### {out['titulo']}\n\n*Tabela `{nome}` — cálculo exato com pandas.*\n\n"
             f"| Indicador | Valor |\n|---|---|\n{linhas}")
    if explicar and out["cards"]:
        fatos = "\n".join(f"- {r}: {v}" for r, v in out["cards"])
        if out["tabela"] is not None:
            fatos += "\n\nTabela calculada:\n" + df_para_exibir(out["tabela"], 15).to_string(index=False)
        explicacao = gerar_llm([
            {"role": "system", "content": (
                "Você é um analista de dados. Explique em português do Brasil, em no máximo 3 frases, o "
                "resultado abaixo, que JÁ FOI CALCULADO. Use somente os números fornecidos, copiando-os "
                "exatamente. Não faça novas contas.")},
            {"role": "user", "content": f"Pergunta: {pergunta}\n\nResultado ({out['titulo']}):\n{fatos}\n\nExplicação:"},
        ], max_tokens=160)
        texto += f"\n\n**🤖 Interpretação:** {explicacao}"
    codigo = f"df = TABELAS['{nome}']\n" + "\n".join(out["codigo"])
    tabela = df_para_exibir(out["tabela"]) if out["tabela"] is not None else None
    return {"texto": texto, "fig": out["fig"], "tabela": tabela, "codigo": codigo, "rotulo": "Resultado calculado"}

## Célula 7 — Interface do usuário
O **Gradio** transforma funções Python em uma aplicação web: cada botão chama uma função, e o retorno preenche os componentes de saída.

- `share=True` gera um link público temporário (72 h), que só funciona enquanto o Colab estiver rodando.
- Para uma aplicação **permanente**, o mesmo código pode ser publicado gratuitamente no **Hugging Face Spaces**.
- Para interromper a aplicação, pare a célula ou rode `demo.close()`.

In [19]:
# ==============================================================================
# CÉLULA 7 — INTERFACE Haja Coração (Gradio)
# Identidade visual: saúde, esporte e tecnologia — verde + branco.
# ==============================================================================
import gradio as gr

TIPOS_ACEITOS = [".pdf", ".docx", ".doc", ".txt", ".csv", ".tsv", ".xlsx"]

def opcoes_tabelas():
    return [("Automático", "auto")] + [(n, n) for n in TABELAS]

def listar_base():
    dados = colecao.get(include=["metadatas"])
    if not dados["ids"]:
        return pd.DataFrame(columns=["Arquivo", "Tipo", "Setor", "Fragmentos"])
    base = pd.DataFrame([{"Arquivo": m.get("arquivo_origem", "?"), "Tipo": m.get("tipo", "texto"),
                          "Setor": m.get("setor", "?")} for m in dados["metadatas"]])
    return base.groupby(["Arquivo", "Tipo", "Setor"]).size().reset_index(name="Fragmentos")

def arquivos_na_base():
    return sorted(listar_base()["Arquivo"].unique().tolist())

def atualizacoes_da_base():
    return (listar_base(),
            gr.update(choices=arquivos_na_base(), value=None),
            gr.update(choices=opcoes_tabelas(), value="auto"))

def acao_ingerir(caminhos, setor):
    if not caminhos:
        return ("⚠️ Selecione pelo menos um arquivo.", gr.update(visible=False)) + atualizacoes_da_base()
    caminhos = caminhos if isinstance(caminhos, list) else [caminhos]
    linhas, previa = [], None
    for caminho in caminhos:
        caminho = getattr(caminho, "name", caminho)
        nome = os.path.basename(caminho)
        try:
            r = ingerir_caminho(caminho, setor)
            if r["df"] is not None:
                df = r["df"]
                tipos = ", ".join(f"`{c}` ({tipo_coluna(df[c])})" for c in df.columns)
                sugestoes = " · ".join(f"*{x}*" for x in sugerir_perguntas(df)[:4])
                linhas.append(f"✅ **{nome}** — {len(df)} linhas × {len(df.columns)} colunas.  \nColunas: {tipos}  \nSugestões: {sugestoes}")
                previa = df_para_exibir(df, 10)
            else:
                linhas.append(f"✅ **{nome}** — {r['fragmentos']} fragmento(s) adicionados à busca.")
        except Exception as e:
            linhas.append(f"❌ **{nome}** — {e}")
    linhas.append(f"\n**Base atual:** {colecao.count()} fragmentos.")
    prev = gr.update(value=previa, visible=previa is not None)
    return ("\n\n".join(linhas), prev) + atualizacoes_da_base()

def acao_exemplos():
    n = carregar_artigos_exemplo()
    return (f"✅ {n} documentos de exemplo adicionados. Total: {colecao.count()} fragmentos.",
            gr.update(visible=False)) + atualizacoes_da_base()

def acao_remover(nome):
    if nome:
        colecao.delete(where={"arquivo_origem": nome})
        if nome in TABELAS:
            del TABELAS[nome]
            caminho = os.path.join(PASTA_TABELAS, re.sub(r"[^\w.]+", "_", nome) + ".pkl")
            if os.path.exists(caminho):
                os.remove(caminho)
    return atualizacoes_da_base()

def acao_perguntar(pergunta, tabela_alvo, setor, explicar):
    if not pergunta or not pergunta.strip():
        return "Digite sua pergunta para começar.", gr.update(visible=False), gr.update(visible=False), gr.update(visible=False)
    try:
        r = responder(pergunta.strip(), tabela_alvo, setor, explicar)
    except Exception as e:
        return (f"❌ Não foi possível concluir a consulta: `{e}`",
                gr.update(visible=False), gr.update(visible=False), gr.update(visible=False))
    fig = gr.update(value=r["fig"], visible=r["fig"] is not None)
    tab = gr.update(value=r["tabela"], visible=r["tabela"] is not None, label=r["rotulo"])
    cod = gr.update(value=r["codigo"], visible=bool(r["codigo"]))
    if r["fig"] is not None:
        plt.close(r["fig"])
    return r["texto"], fig, tab, cod





CSS_HAJA = """

/* ==========================================
   HAJA CORAÇÃO — PROFESSIONAL UI
   Minimalista | Verde | Branco
========================================== */

:root {
    --hc-primary: #16834A;
    --hc-primary-dark: #105C36;
    --hc-primary-light: #EAF5EE;

    --hc-background: #F7F9F8;
    --hc-surface: #FFFFFF;
    --hc-text: #172B22;
    --hc-muted: #68786F;
    --hc-border: #E4EAE6;

    --hc-radius: 12px;
}

/* ==========================================
   FUNDO E ESTRUTURA
========================================== */

html, body {
    background: var(--hc-background) !important;
    color: var(--hc-text) !important;
    color-scheme: light !important;
}

.gradio-container {
    background: var(--hc-background) !important;
    color: var(--hc-text) !important;
    max-width: 1440px !important;
    margin: auto !important;

    font-family: Inter, "Segoe UI", Arial, sans-serif !important;
}

/* Variáveis de tema do Gradio */
.gradio-container,
.gradio-container .contain {
    --background-fill-primary: #FFFFFF !important;
    --background-fill-secondary: #F7F9F8 !important;
    --background-fill-tertiary: #EFF3F0 !important;

    --block-background-fill: #FFFFFF !important;
    --block-border-color: #E4EAE6 !important;

    --body-background-fill: #F7F9F8 !important;
    --body-text-color: #172B22 !important;
    --body-text-color-subdued: #68786F !important;

    --input-background-fill: #FFFFFF !important;
    --input-border-color: #DCE5DF !important;

    --button-primary-background-fill: #16834A !important;
    --button-primary-background-fill-hover: #105C36 !important;
    --button-primary-text-color: #FFFFFF !important;
}

/* ==========================================
   CABEÇALHO
========================================== */

#hc-header {
    background: linear-gradient(
        115deg,
        #105C36 0%,
        #16834A 100%
    ) !important;

    padding: 38px 36px !important;
    border-radius: 16px !important;
    margin-bottom: 28px !important;

    box-shadow: 0 8px 24px rgba(16, 92, 54, 0.10);
}

#hc-header h1 {
    color: #FFFFFF !important;
    font-size: 34px !important;
    font-weight: 800 !important;
    letter-spacing: -1px !important;
}

#hc-header p {
    color: #E7F5EB !important;
    font-size: 15px !important;
    line-height: 1.7 !important;
}

/* ==========================================
   TÍTULOS E TEXTOS
========================================== */

.gradio-container h1,
.gradio-container h2,
.gradio-container h3 {
    color: #173D2A !important;
    font-weight: 750 !important;
    letter-spacing: -0.4px;
}

.gradio-container p,
.gradio-container label,
.gradio-container span {
    color: #344A3C;
}

/* Textos secundários */
.gradio-container .prose,
.gradio-container .markdown {
    color: #344A3C !important;
    line-height: 1.7 !important;
}

/* ==========================================
   PAINÉIS E CARTÕES
========================================== */

.gradio-container .block,
.gradio-container .panel,
.gradio-container .form,
.gradio-container .wrap {
    background-color: #FFFFFF !important;
    color: #172B22 !important;
    border-color: #E4EAE6 !important;
}

.gradio-container .block {
    border-radius: 12px !important;
}

.gradio-container .panel {
    border-radius: 14px !important;
    box-shadow: 0 2px 10px rgba(20, 50, 30, 0.035);
}

/* Remove fundos azul-escuros dos agrupamentos */
.gradio-container .form,
.gradio-container [data-testid="block-info"],
.gradio-container [data-testid="tabitem"] {
    background: #FFFFFF !important;
    color: #172B22 !important;
}

/* ==========================================
   CAMPOS DE ENTRADA
========================================== */

.gradio-container input,
.gradio-container textarea,
.gradio-container select,
.gradio-container .input-container {
    background: #FFFFFF !important;
    color: #172B22 !important;
    border: 1px solid #DCE5DF !important;
    border-radius: 10px !important;
    box-shadow: none !important;
}

.gradio-container input::placeholder,
.gradio-container textarea::placeholder {
    color: #84938A !important;
    opacity: 1 !important;
}

.gradio-container input:focus,
.gradio-container textarea:focus {
    border-color: #16834A !important;
    box-shadow: 0 0 0 3px rgba(22, 131, 74, 0.10) !important;
}

/* Área de pergunta */
.gradio-container textarea {
    min-height: 110px !important;
    padding: 15px !important;
    line-height: 1.6 !important;
}

/* ==========================================
   BOTÕES
========================================== */

.gradio-container button {
    border-radius: 9px !important;
    font-weight: 650 !important;
    transition: all 0.2s ease !important;
}

.gradio-container button.primary {
    background: #16834A !important;
    color: #FFFFFF !important;
    border: 1px solid #16834A !important;
    box-shadow: 0 3px 8px rgba(22, 131, 74, 0.12);
}

.gradio-container button.primary:hover {
    background: #105C36 !important;
    transform: translateY(-1px);
}

.gradio-container button.secondary {
    background: #FFFFFF !important;
    color: #344A3C !important;
    border: 1px solid #DCE5DF !important;
}

.gradio-container button.secondary:hover {
    background: #EAF5EE !important;
    color: #105C36 !important;
}

/* Botões desabilitados */
.gradio-container button:disabled {
    opacity: 0.55 !important;
    cursor: not-allowed !important;
}

/* ==========================================
   ABAS
========================================== */

.gradio-container .tab-nav {
    background: transparent !important;
    border-bottom: 1px solid #E4EAE6 !important;
    gap: 10px !important;
}

.gradio-container .tab-nav button {
    background: transparent !important;
    color: #68786F !important;
    border: none !important;
    border-radius: 0 !important;
    padding: 13px 18px !important;
}

.gradio-container .tab-nav button.selected {
    color: #16834A !important;
    border-bottom: 3px solid #16834A !important;
    font-weight: 750 !important;
}

/* ==========================================
   TABELAS E RESULTADOS
========================================== */

.gradio-container table,
.gradio-container .table-wrap,
.gradio-container .dataframe {
    background: #FFFFFF !important;
    color: #172B22 !important;
    border: 1px solid #E4EAE6 !important;
    border-radius: 10px !important;
    overflow: hidden !important;
}

.gradio-container th {
    background: #EAF5EE !important;
    color: #105C36 !important;
    font-weight: 750 !important;
    padding: 14px !important;
    border-color: #DCE8DF !important;
}

.gradio-container td {
    background: #FFFFFF !important;
    color: #344A3C !important;
    padding: 13px !important;
    border-color: #E8EEE9 !important;
}

.gradio-container tr:hover td {
    background: #F5FAF6 !important;
}

/* ==========================================
   MARKDOWN E RESPOSTAS DA IA
========================================== */

.gradio-container .prose h1,
.gradio-container .prose h2,
.gradio-container .prose h3 {
    color: #105C36 !important;
    margin-top: 20px !important;
}

.gradio-container .prose strong {
    color: #105C36 !important;
    font-weight: 750 !important;
}

.gradio-container .prose blockquote {
    background: #EAF5EE !important;
    border-left: 4px solid #16834A !important;
    padding: 14px 18px !important;
    border-radius: 0 9px 9px 0 !important;
}

.gradio-container .prose code {
    background: #EFF5F0 !important;
    color: #105C36 !important;
    border-radius: 5px !important;
}

/* ==========================================
   CAIXAS INFORMATIVAS
========================================== */

.hc-info {
    background: #EFF7F1 !important;
    border: 1px solid #D9EBDD !important;
    border-radius: 12px !important;
    padding: 18px 20px !important;
    color: #294A36 !important;
    line-height: 1.7 !important;
}

/* ==========================================
   CHECKBOX E SELETORES
========================================== */

.gradio-container input[type="checkbox"] {
    accent-color: #16834A !important;
}

.gradio-container [role="checkbox"],
.gradio-container [role="radio"] {
    accent-color: #16834A !important;
}

/* ==========================================
   RODAPÉ E ACABAMENTO
========================================== */

footer {
    display: none !important;
}

::selection {
    background: #C8EAD2 !important;
    color: #105C36 !important;
}

/* Scrollbar */
.gradio-container ::-webkit-scrollbar {
    width: 8px;
    height: 8px;
}

.gradio-container ::-webkit-scrollbar-thumb {
    background: #C7D8CC;
    border-radius: 10px;
}

.gradio-container ::-webkit-scrollbar-track {
    background: #F7F9F8;
}

"""

with gr.Blocks(
    title="Haja Coração | Inteligência Esportiva",
    theme=gr.themes.Soft(
        primary_hue="green",
        secondary_hue="emerald",
        neutral_hue="slate"
    ),
    css=CSS_HAJA
) as demo:
    gr.HTML("""
    <div id="hc-header">
      <div class="hc-kicker">Inteligência esportiva • Saúde • Dados</div>
      <h1>♥ Haja Coração</h1>
      <p>Assistente inteligente para análise cardiovascular, monitoramento de atletas e apoio ao desempenho esportivo.</p>
    </div>
    """)
    gr.Markdown(
        "<div class='hc-info'>"
        "**Como funciona:** envie documentos e tabelas para a base de conhecimento. "
        "Depois, faça perguntas sobre fisiologia, desempenho, arquitetura ou seus dados. "
        "O sistema usa recuperação de contexto com DeepSeek-R1 e cálculos determinísticos com Pandas."
        "</div>"
    )

    with gr.Tab("💬 Assistente"):
        gr.Markdown("### Converse com o Haja Coração")
        gr.Markdown("Faça uma pergunta sobre o projeto ou solicite uma análise dos dados dos atletas.")
        entrada_pergunta = gr.Textbox(
            lines=3, label="Sua pergunta",
            placeholder="Ex.: Como o monitoramento da frequência cardíaca pode ajudar a identificar sinais de sobrecarga?"
        )
        with gr.Row():
            entrada_tabela = gr.Dropdown(choices=opcoes_tabelas(), value="auto", label="Fonte de dados", scale=2)
            entrada_filtro = gr.Dropdown(
                choices=[("Todos os setores", "todos")] + SETORES,
                value="todos", label="Área de conhecimento", scale=2
            )
        entrada_explicar = gr.Checkbox(value=True, label="Gerar interpretação dos resultados com DeepSeek")
        with gr.Row():
            botao_perguntar = gr.Button("🔎 Analisar pergunta", variant="primary", scale=4)
            botao_limpar = gr.ClearButton(
                [entrada_pergunta], value="Limpar pergunta", scale=1
            )
        gr.Examples(
            label="Perguntas para começar",
            inputs=entrada_pergunta,
            examples=[
                "Por que os exames periódicos não bastam para acompanhar o atleta?",
                "Quais são os objetivos do monitoramento cardiovascular?",
                "Como a sobrecarga física pode afetar o desempenho?",
                "Quais tecnologias fazem parte da arquitetura do projeto?",
                "Quais cuidados de segurança e privacidade são previstos?"
            ]
        )
        saida_resposta = gr.Markdown()
        saida_grafico = gr.Plot(label="Visualização dos dados", visible=False)
        saida_tabela = gr.Dataframe(label="Fontes e dados utilizados", visible=False, interactive=False, wrap=True)
        saida_codigo = gr.Code(label="Código Pandas equivalente", language="python", visible=False)

    with gr.Tab("📂 Documentos"):
        gr.Markdown("### Alimente a base de conhecimento")
        gr.Markdown("Adicione PDFs, documentos de texto ou planilhas. Escolha a área temática para facilitar a recuperação.")
        with gr.Row():
            with gr.Column(scale=3):
                entrada_arquivos = gr.File(
                    label="Selecione os arquivos",
                    file_count="multiple", file_types=TIPOS_ACEITOS, type="filepath"
                )
            with gr.Column(scale=2):
                entrada_setor = gr.Dropdown(choices=SETORES, value="geral", label="Classificação do conteúdo")
                botao_ingerir = gr.Button("＋ Processar documentos", variant="primary")
                botao_exemplos = gr.Button("📚 Carregar documentos demonstrativos")
        saida_status = gr.Markdown()
        saida_previa = gr.Dataframe(label="Prévia da tabela importada", visible=False, interactive=False)

    with gr.Tab("🗃️ Base de conhecimento"):
        gr.Markdown("### Conteúdo indexado")
        gr.Markdown("Consulte os arquivos e fragmentos disponíveis para o assistente.")
        saida_base = gr.Dataframe(value=listar_base(), label="Documentos armazenados", interactive=False)
        with gr.Row():
            entrada_remover = gr.Dropdown(choices=arquivos_na_base(), label="Documento para remoção", scale=3)
            botao_remover = gr.Button("Remover documento", variant="stop", scale=1)
            botao_atualizar = gr.Button("Atualizar base", scale=1)
        gr.Markdown(
            "<div class='hc-info'>Dica: mantenha os documentos organizados por setor e evite arquivos "
            "duplicados ou com conteúdo desatualizado para melhorar a qualidade da recuperação.</div>"
        )

    saidas_base = [saida_base, entrada_remover, entrada_tabela]
    botao_ingerir.click(acao_ingerir, [entrada_arquivos, entrada_setor],
                        [saida_status, saida_previa] + saidas_base)
    botao_exemplos.click(acao_exemplos, None, [saida_status, saida_previa] + saidas_base)
    botao_remover.click(acao_remover, entrada_remover, saidas_base)
    botao_atualizar.click(atualizacoes_da_base, None, saidas_base)

    saidas_resposta = [saida_resposta, saida_grafico, saida_tabela, saida_codigo]
    entradas_pergunta = [entrada_pergunta, entrada_tabela, entrada_filtro, entrada_explicar]
    botao_perguntar.click(acao_perguntar, entradas_pergunta, saidas_resposta)
    entrada_pergunta.submit(acao_perguntar, entradas_pergunta, saidas_resposta)

demo.launch(share=True, debug=False)


/tmp/ipykernel_572/3006987238.py:449: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e79ceba47865d37f36.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
